<div dir="rtl">

# ارزیابی و انتخاب مدل (Model Evaluation and Selection)

در این نوت‌بوک، مباحث پیشرفته و کلیدی ارزیابی عملکرد مدل‌های یادگیری ماشین، تکنیک‌های اعتبارسنجی، بهینه‌سازی فراپارامترها و انتخاب مدل نهایی به صورت کاملاً کاربردی و عملی پیاده‌سازی شده‌اند.

---
</div>

<div dir="rtl">

## ۳۱. ماتریس درهم‌ریختگی (Confusion Matrix) - تفسیر عملی
ماتریس درهم‌ریختگی ابزاری پایه‌ای برای ارزیابی عملکرد مدل‌های دسته‌بندی (Classification) است که مقادیر پیش‌بینی‌شده را در برابر مقادیر واقعی قرار می‌دهد:
- **True Positive (TP):** نمونه‌های مثبتی که درست پیش‌بینی شده‌اند.
- **True Negative (TN):** نمونه‌های منفی که درست پیش‌بینی شده‌اند.
- **False Positive (FP):** خطای نوع اول (خطای آلفا) - پیش‌بینی مثبت اشتباه.
- **False Negative (FN):** خطای نوع دوم (خطای بتا) - پیش‌بینی منفی اشتباه.

</div>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# تولید داده‌های نمونه
X, y = make_classification(n_samples=1000, n_features=20, n_classes=2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# آموزش مدل رگرسیون لجستیک
model = LogisticRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# رسم ماتریس درهم‌ریختگی
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm, display_labels=["Negative", "Positive"]
)
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix - Practical Example")
plt.show()

<div dir="rtl">

## ۳۲. معیارهای ارزیابی دسته‌بندی (Classification Metrics)
برای مقایسه دقیق مدل‌ها تنها به دقت (Accuracy) اتکا نمی‌کنیم، زیرا در داده‌های نامتوازن (Imbalanced Data) کارایی ندارد. معیارهای اصلی عبارتند از:
- **Precision (دقت مثبت):** نسبت پیش‌بینی‌های مثبت درست به کل پیش‌بینی‌های مثبت $\frac{TP}{TP + FP}$.
- **Recall / Sensitivity (فراخوانی):** نسبت نمونه‌های مثبت واقعی که درست شناسایی شده‌اند $\frac{TP}{TP + FN}$.
- **F1-Score:** میانگین همساز (Harmonic Mean) بین Precision و Recall.

</div>

In [ ]:
from sklearn.metrics import (
    classification_report,
    precision_score,
    recall_score,
    f1_score,
)

# محاسبه جامع معیارها
print(classification_report(y_test, y_pred))

print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred):.4f}")

<div dir="rtl">

## ۳۳. منحنی ROC و سطح زیر آن (AUC)
منحنی ROC رابطه بین **True Positive Rate (Sensitivity)** و **False Positive Rate (1 - Specificity)** را در آستانه‌های مختلف تصمیم‌گیری نمایش می‌دهد. معیار **AUC** (Area Under Curve) قدرت تفکیک‌کنندگی مدل را از صفر تا یک نشان می‌دهد.

</div>

In [ ]:
from sklearn.metrics import roc_curve, auc

y_prob = model.predict_proba(X_test)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC curve (AUC = {roc_auc:.2f})")
plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Receiver Operating Characteristic (ROC)")
plt.legend(loc="lower right")
plt.show()

<div dir="rtl">

## ۳۴. اعتبارسنجی متقاطع K-Fold
برای جلوگیری از وابستگی ارزیابی به یک تقسیم‌بندی خاص (Train/Test Split)، داده‌ها به $K$ بخش (Fold) تقسیم می‌شوند. مدل $K$ بار آموزش می‌بیند که در هر بار یک بخش به عنوان تست و مابقی برای آموزش استفاده می‌شوند.

</div>

In [ ]:
from sklearn.model_selection import KFold, cross_val_score

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X, y, cv=kf, scoring="accuracy")

print(f"K-Fold CV Scores: {cv_scores}")
print(f"Mean Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

<div dir="rtl">

## ۳۵. اعتبارسنجی متقاطع لایه‌بندی‌شده (Stratified K-Fold)
در مسائل دسته‌بندی با توزیع نامتوازن کلاس‌ها، K-Fold معمولی ممکن است باعث شود برخی Foldها فاقد نمونه‌های یک کلاس باشند. **Stratified K-Fold** تضمین می‌کند که نسبت کلاس‌ها در هر Fold دقیقاً مشابه کل مجموعه داده باشد.

</div>

In [ ]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
stratified_scores = cross_val_score(model, X, y, cv=skf, scoring="accuracy")

print(f"Stratified K-Fold Scores: {stratified_scores}")
print(
    f"Mean Accuracy: {stratified_scores.mean():.4f} (+/- {stratified_scores.std() * 2:.4f})"
)

<div dir="rtl">

## ۳۶. نمونه‌برداری بوسترپ (Bootstrap - درک مفهومی)
بوسترپ تکنیکی آماری است که در آن با جایگذاری (Sampling with Replacement) از روی مجموعه داده اصلی، نمونه‌های جدیدی با همان اندازه ایجاد می‌شود. این روش برای تخمین فاصله اطمینان (Confidence Interval) و در الگوریتم‌هایی مثل Random Forest کاربرد دارد.

</div>

In [ ]:
from sklearn.utils import resample

# شبیه‌سازی نمونه‌برداری بوسترپ
bootstrapped_sample = resample(
    X_train, n_samples=len(X_train), replace=True, random_state=42
)
print(
    f"Original shape: {X_train.shape}, Bootstrapped shape: {bootstrapped_sample.shape}"
)

<div dir="rtl">

## ۳۷. منحنی‌های یادگیری (Learning Curves)
منحنی یادگیری عملکرد مدل بر روی داده‌های آموزشی و اعتبارسنجی را بر اساس حجم داده‌های آموزشی نشان می‌دهد. این نمودار برای تشخیص بیش‌برازش (Overfitting) و کم‌برازش (Underfitting) بسیار مفید است.

</div>

In [ ]:
from sklearn.model_selection import learning_curve

train_sizes, train_scores, val_scores = learning_curve(
    model,
    X,
    y,
    cv=5,
    scoring="accuracy",
    train_sizes=np.linspace(0.1, 1.0, 10),
    random_state=42,
)

train_mean = np.mean(train_scores, axis=1)
val_mean = np.mean(val_scores, axis=1)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_mean, label="Training score", marker="o")
plt.plot(train_sizes, val_mean, label="Cross-validation score", marker="o")
plt.xlabel("Training Set Size")
plt.ylabel("Accuracy Score")
plt.title("Learning Curve")
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

<div dir="rtl">

## ۳۸. منحنی‌های اعتبارسنجی (Validation Curves)
برخلاف منحنی یادگیری که حجم داده را بررسی می‌کند، منحنی اعتبارسنجی تاثیر تغییر مقدار یک فراپارامتر خاص (مانند C در رگرسیون لجستیک) را بر خطای آموزش و اعتبارسنجی نمایش می‌دهد.

</div>

In [ ]:
from sklearn.model_selection import validation_curve

param_range = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
train_scores, val_scores = validation_curve(
    LogisticRegression(max_iter=1000),
    X,
    y,
    param_name="C",
    param_range=param_range,
    cv=5,
    scoring="accuracy",
)

train_mean = np.mean(train_scores, axis=1)
val_mean = np.mean(val_scores, axis=1)

plt.figure(figsize=(8, 5))
plt.plot(param_range, train_mean, label="Training score", marker="o")
plt.plot(param_range, val_mean, label="Cross-validation score", marker="o")
plt.xscale("log")
plt.xlabel("Parameter C")
plt.ylabel("Accuracy Score")
plt.title("Validation Curve for Logistic Regression (C)")
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

<div dir="rtl">

## ۳۹. تحلیل خطا (Error Analysis)
تحلیل خطا یعنی بررسی نمونه‌هایی که مدل به اشتباه پیش‌بینی کرده است (False Positives و False Negatives) تا الگوها، کمبودهای داده یا ویژگی‌های گم‌شده را شناسایی کنیم.

</div>

In [ ]:
# یافتن نمونه‌های خطای پیش‌بینی
misclassified_idx = np.where(y_pred != y_test)[0]
print(f"تعداد کل خطاها در مجموعه تست: {len(misclassified_idx)}")
print(f"ایندکس نمونه‌های اشتباه: {misclassified_idx[:10]} (نمایش ۱۰ مورد اول)")

<div dir="rtl">

## ۴۰. پارامترها در برابر فراپارامترها (Parameters vs Hyperparameters)
- **پارامترها (Parameters):** متغیرهایی درون مدل که طی فرآیند آموزش (Training) از روی داده‌ها یاد گرفته می‌شوند (مثل ضرایب وزن‌ها $w$ و بایاس $b$).
- **فراپارامترها (Hyperparameters):** تنظیماتی که قبل از آموزش توسط طراح تعیین می‌شوند و توسط الگوریتم یادگیری به صورت خودکار به دست نمی‌آیند (مثل نرخ یادگیری، تعداد درختان در Random Forest یا مقدار C).

</div>

<div dir="rtl">

## ۴۱. جستجوی شبکه‌ای (GridSearchCV)
تکنیکی برای یافتن بهترین ترکیب فراپارامترها با بررسی تمام حالت‌های ممکن در یک فضای جستجوی تعریف‌شده به همراه اعتبارسنجی متقاطع.

</div>

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "C": [0.01, 0.1, 1.0, 10.0],
    "penalty": ["l2"],
    "solver": ["lbfgs", "liblinear"],
}

grid_search = GridSearchCV(
    estimator=LogisticRegression(max_iter=1000),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
)
grid_search.fit(X_train, y_train)

print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best CV Score: {grid_search.best_score_:.4f}")

<div dir="rtl">

## ۴۲. جستجوی تصادفی (RandomizedSearchCV)
هنگامی که فضای جستجوی فراپارامترها بسیار بزرگ است، GridSearchCV بسیار کند خواهد بود. **RandomizedSearchCV** تعداد مشخصی ترکیب تصادفی را امتحان می‌کند که کارایی بسیار بالاتری دارد.

</div>

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform

param_dist = {
    "C": uniform(loc=0, scale=10),
    "penalty": ["l2"],
    "solver": ["lbfgs", "liblinear"],
}

random_search = RandomizedSearchCV(
    estimator=LogisticRegression(max_iter=1000),
    param_distributions=param_dist,
    n_iter=10,
    cv=5,
    scoring="accuracy",
    random_state=42,
)
random_search.fit(X_train, y_train)

print(f"Best Random Parameters: {random_search.best_params_}")
print(f"Best Random CV Score: {random_search.best_score_:.4f}")

<div dir="rtl">

## ۴۳. جلوگیری از نشت داده در تیونینگ مدل (Avoiding Data Leakage)
**نشت داده (Data Leakage)** زمانی رخ می‌دهد که اطلاعات مجموعه تست یا اعتبارسنجی ناخواسته وارد فرآیند آموزش یا پیش‌پردازش شوند. برای جلوگیری از این خطا، استفاده از **Pipeline** الزامی است تا تمام عملیات مقیاس‌بندی و تبدیل صرفاً روی داده‌های آموزشی هر Fold انجام شود.

</div>

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# ساخت پایپ‌لاین برای جلوگیری از نشت داده
pipeline = Pipeline(
    [("scaler", StandardScaler()), ("classifier", LogisticRegression())]
)

pipeline.fit(X_train, y_train)
print(f"Pipeline Test Accuracy: {pipeline.score(X_test, y_test):.4f}")

<div dir="rtl">

## ۴۴. انتخاب مدل نهایی (Final Model Selection)
پس از مقایسه چندین الگوریتم مختلف (مثلاً رگرسیون لجستیک، درخت تصمیم، ماشین بردار پشتیبان) به همراه بهینه‌سازی فراپارامترها با استفاده از Pipeline، بهترین مدل انتخاب شده و روی کل داده‌های آموزشی نهایی آموزش داده می‌شود و عملکرد نهایی آن روی مجموعه Test نهایی ارزیابی می‌گردد.

</div>

In [ ]:
# ارزیابی نهایی مدل برگزیده
best_model = grid_search.best_estimator_
final_accuracy = best_model.score(X_test, y_test)
print(f"Final Selected Model Test Accuracy: {final_accuracy:.4f}")